# 16 - USS Auto-Labeler Evaluation  (IV 2027 paper track)

**Separate paper track - not part of WACV 2660.** Scores the Bosch USS-geometry
auto-labeler (prototype by M. K. Uludag, Aug 2024; ported in
`src/vision_uss_research/autolabel/`) against the human **Label V2 polygons**
(`PolygonX/PolygonY`, world mm) that ship with every corpus sequence. Frozen
WACV numbers are untouched; outputs go to `autolabel/al_*` runs.

### What the prototype really does (from its source, not the report)
Picks one production USS map object slot (`Map_Obj{n}_P1/P2`), takes its final
x-offset only, projects it along the final ego yaw, and draws a **fixed
0.75 x 1.75 m box**. `v1` below reproduces the prototype's saved 25-trace
evaluation exactly (0.1739 m / 0.7170 m on two second-programme traces).

### Questions this notebook answers
1. **Per-class label quality** at corpus scale: IoU, IoU>=0.5 rate, one-to-one
   corner error, centroid (size-free) error - with bootstrap CIs.
2. **Is "object height determines quality" real, or a size artefact?** The fixed
   template cannot fit small or large footprints. `*_oracle_size_iou` re-centres the
   human polygon's own shape on the predicted centre: the gap to the real IoU is
   the template-size penalty. Low vs High is then compared within GT-area strata.
3. **Which single fix helps** (ablations, each one change vs v1): lateral offset
   (both signs - Map_Obj y convention undocumented), time sync, track-based slot
   selection, segment-sized box, depth shift (box behind the reflecting face).
4. **v2 = learned size prior** (section 5): lateral + per-class box dimensions
   learned on held-out folds (fair for staged data: the campaign metadata names
   the object), centred on the map object (and, as an ablation, depth-shifted).
5. **Placement** (section 6): central / 45 deg / displaced approach from the
   metadata tags.

Run history: `al_eval_v1` (smoke, 300 seqs, 2026-09-30) fixed the lateral sign
(+1: centroid error 0.50 -> 0.27 m) and ruled out sync / sel2 / seg boxes.
`al_eval_v2` adds the depth-shift variant and the stored face geometry.

Evaluated against the FIRST low/high target row of each sequence; the headline
analysis uses single-target sequences only (`n_targets == 1`) so the target is
unambiguous.

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "asammdf"], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import csv
import json as _json
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from vision_uss_research.autolabel.uss_box import (FIXED_SIZE_VARIANTS, VARIANTS,
                                                   evaluate_sequence)
from vision_uss_research.csv_io import read_csv_tolerant, repair_csv
from vision_uss_research.runs import finish_run, new_run_id, start_run
from vision_uss_research.sequences import sequence_files, with_drive_retry

MAX_SEQUENCES = 300       # smoke first; 0 = every sequence with MF4 + Label V2
NUM_WORKERS = 8
SEED = 0
N_BOOT = 2000
IOU_OK = 0.5

RUN_DIR = ARTIFACTS / "autolabel" / new_run_id("al", "eval_v2")
RESULTS_CSV = RUN_DIR / "autolabel_eval.csv"
RUN_DIR.mkdir(parents=True, exist_ok=True)

METRICS = ["iou", "corner_err_nn_m", "corner_err_matched_m", "centroid_err_m", "pred_area_m2"]
FIELDS = (["sequence_id", "error", "n_targets", "object", "height_bin", "gt_area_m2",
           "gt_n_vertices", "n_slots", "slot_v1", "slot_v1_score_mm", "slot_v2",
           "slot_v2_score_mm"]
          + [f"{v}_{m}" for v in VARIANTS for m in METRICS]
          + [f"{v}_oracle_size_iou" for v in FIXED_SIZE_VARIANTS]
          + ["face_cx", "face_cy", "ego_yaw_rad", "obj_side", "gt_len_along_m",
             "gt_wid_across_m", "approach", "driving_direction", "gt_polygon"])
TEXT_FIELDS = {"sequence_id", "error", "object", "height_bin", "approach",
               "driving_direction", "gt_polygon"}

seq_ids = sorted(p.name for p in with_drive_retry(lambda: list(RAW_ROOT.iterdir()))
                 if p.is_dir())
print(f"sequence folders: {len(seq_ids)}")
if MAX_SEQUENCES and MAX_SEQUENCES < len(seq_ids):
    rng = np.random.default_rng(SEED)
    seq_ids = sorted(rng.choice(seq_ids, MAX_SEQUENCES, replace=False).tolist())

# resume: only error-free rows count as done; errored ones are retried
done = set()
if RESULTS_CSV.exists():
    repair_csv(RESULTS_CSV, FIELDS)
    prev = read_csv_tolerant(RESULTS_CSV, FIELDS)
    prev["error"] = prev["error"].fillna("")
    keep = prev[(prev["error"] == "") | prev["error"].isin(
        ["no_target_polygon", "no_object_selected", "no_mf4_or_label"])]
    keep = keep.drop_duplicates("sequence_id", keep="last")
    keep.to_csv(RESULTS_CSV, index=False)
    done = set(keep["sequence_id"].astype(str))
    print(f"resuming: {len(done)} final rows kept, {prev['sequence_id'].nunique() - len(done)} to retry")
todo = [s for s in seq_ids if s not in done]

start_run(RUN_DIR, config={"max_sequences": MAX_SEQUENCES, "seed": SEED,
                           "variants": {k: [s, kw] for k, (s, kw) in VARIANTS.items()},
                           "iou_ok": IOU_OK},
          inputs=["raw/vision_uss_sequences"], repo_root=REPO_DIR)
print(f"to process: {len(todo)}\noutput    : {RUN_DIR}")

## Run the auto-labeler on every sequence (parallel, resumable)

In [ ]:
def metadata_tags(meta_path):
    """Approach (central / 45deg / displaced) and driving direction tags."""
    out = {"approach": "", "driving_direction": ""}
    if meta_path is None:
        return out
    try:
        tags = _json.loads(meta_path.read_text()).get("tags", [])
    except Exception:
        return out
    for t in tags:
        sub = str(t.get("tagSubCategory", {}).get("name", "")).lower()
        if sub == "approach":
            out["approach"] = str(t.get("name", "")).replace("°", "deg").strip().lower()
        elif sub == "driving direction":
            out["driving_direction"] = str(t.get("name", "")).strip().lower()
    return out

def process(seq_id):
    seq_dir = RAW_ROOT / seq_id
    try:
        found = with_drive_retry(sequence_files, seq_dir)
        mf4 = next(iter(sorted(seq_dir.glob("*.mf4"))), None)
        if mf4 is None or found.get("label_v2_csv") is None:
            return {"sequence_id": seq_id, "error": "no_mf4_or_label"}
        row = evaluate_sequence(mf4, found["label_v2_csv"])
        row.update(metadata_tags(found.get("metadata_json")))
    except Exception as e:
        row = {"error": f"{type(e).__name__}:{e}"}
    row["sequence_id"] = seq_id
    return row

def append(path, rows, fields):
    header = not path.exists()
    with path.open("a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        if header:
            w.writeheader()
        w.writerows(rows)

if todo:
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        futs = [pool.submit(process, s) for s in todo]
        for fut in tqdm(as_completed(futs), total=len(futs), desc="Auto-label", unit="seq"):
            append(RESULTS_CSV, [fut.result()], FIELDS)

res = read_csv_tolerant(RESULTS_CSV, FIELDS)
for c in FIELDS:
    if c not in TEXT_FIELDS:
        res[c] = pd.to_numeric(res[c], errors="coerce")
res["error"] = res["error"].fillna("")
print(f"rows: {len(res)}")
print(res["error"].replace("", "OK").str.split(":").str[0].value_counts().to_string())

## Evaluation set

Canonical object names come from `configs/datasets/label_v2_class_mapping.yaml`
(same aliasing as notebooks 01/11). Headline set = scored, single-target.

In [ ]:
import re
import yaml

def _norm(s):
    return re.sub(r"[^a-z0-9]+", "", str(s).lower())
_alias = {}
_m = yaml.safe_load((REPO_DIR / "configs" / "datasets" / "label_v2_class_mapping.yaml").read_text())
for canon, variants in (_m.get("canonical_objects") or {}).items():
    for v in [canon, *variants]:
        _alias[_norm(v)] = canon
res["obj"] = res["object"].map(lambda o: _alias.get(_norm(o), str(o).strip().lower()))

scored = res[(res["error"] == "") & res["v1_iou"].notna()].copy()
ev = scored[scored["n_targets"] == 1].copy()
print(f"scored: {len(scored)}   single-target (headline): {len(ev)}")
print(ev["height_bin"].value_counts().to_string())
print(f"v1 and sel2 picked the same slot: {(ev['slot_v1'] == ev['slot_v2']).mean():.1%}")

## 1 - Per-class label quality (v1 as-run), bootstrap 95% CIs

In [ ]:
rng = np.random.default_rng(SEED)

def boot_ci(x, fn=np.median, n=N_BOOT):
    x = pd.to_numeric(pd.Series(np.asarray(x).ravel()), errors="coerce").dropna()
    x = x.to_numpy(float)
    if len(x) < 5:
        return (np.nan, np.nan)
    b = [fn(x[rng.integers(0, len(x), len(x))]) for _ in range(n)]
    return tuple(np.round(np.percentile(b, [2.5, 97.5]), 3))

def quality_table(df, prefix="v1", by="obj", min_n=10):
    rows = []
    for k, g in df.groupby(by):
        if len(g) < min_n:
            continue
        iou = g[f"{prefix}_iou"]
        rows.append({by: k, "n": len(g), "height_bin": g["height_bin"].mode().iat[0],
                     "gt_area_m2": round(g["gt_area_m2"].median(), 3),
                     "iou_med": round(iou.median(), 3), "iou_med_ci": boot_ci(iou),
                     f"iou>={IOU_OK}": round((iou >= IOU_OK).mean(), 3),
                     "corner_matched_m": round(g[f"{prefix}_corner_err_matched_m"].median(), 3),
                     "corner_nn_m": round(g[f"{prefix}_corner_err_nn_m"].median(), 3),
                     "centroid_m": round(g[f"{prefix}_centroid_err_m"].median(), 3),
                     "oracle_size_iou": round(g[f"{prefix}_oracle_size_iou"].median(), 3)
                                        if f"{prefix}_oracle_size_iou" in g else np.nan})
    out = pd.DataFrame(rows)
    return out.sort_values("iou_med", ascending=False) if len(out) else out

by_obj = quality_table(ev)
by_bin = quality_table(ev, by="height_bin", min_n=1)
by_obj.to_csv(RUN_DIR / "quality_by_object_v1.csv", index=False)
by_bin.to_csv(RUN_DIR / "quality_by_heightbin_v1.csv", index=False)
display(by_bin); display(by_obj)

## 2 - Height effect or size artefact?

If the Low/High gap is mostly template-size penalty, (a) `oracle_size_iou` for
Low and High should be close, and (b) within GT-area strata the Low/High IoU gap
should shrink. Centroid error is size-free: compare it directly.

In [ ]:
ev["size_penalty"] = ev["v1_oracle_size_iou"] - ev["v1_iou"]
ev["area_bin"] = pd.qcut(ev["gt_area_m2"], 4, duplicates="drop")
strat = (ev.groupby(["area_bin", "height_bin"], observed=True)
           .agg(n=("v1_iou", "size"), iou_med=("v1_iou", "median"),
                oracle_med=("v1_oracle_size_iou", "median"),
                centroid_med=("v1_centroid_err_m", "median"))
           .round(3).reset_index())
strat.to_csv(RUN_DIR / "height_vs_size_strata.csv", index=False)
display(strat)
print("size-free check - median centroid error by height bin:")
print(ev.groupby("height_bin")["v1_centroid_err_m"].agg(["size", "median"]).round(3).to_string())
for hb in ("low", "high"):
    print(hb, "centroid err CI:", boot_ci(ev.loc[ev["height_bin"] == hb, "v1_centroid_err_m"]))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
TEMPLATE = 0.75 * 1.75
for hb, col in (("high", "tab:blue"), ("low", "tab:orange")):
    g = ev[ev["height_bin"] == hb]
    axes[0].scatter(g["gt_area_m2"], g["v1_iou"], s=8, alpha=0.5, c=col, label=hb)
    axes[1].scatter(g["gt_area_m2"], g["v1_oracle_size_iou"], s=8, alpha=0.5, c=col, label=hb)
    axes[2].hist(g["v1_centroid_err_m"].clip(upper=3), bins=40, alpha=0.55, color=col, label=hb)
for ax in axes[:2]:
    ax.axvline(TEMPLATE, color="k", ls="--", lw=1); ax.set_xscale("log")
    ax.set_xlabel("human box area (m², log)"); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
axes[0].set_title("v1 IoU (fixed 1.31 m² template, dashed)")
axes[1].set_title("oracle-size IoU (localisation only)")
axes[2].set_title("centroid error (m, clipped at 3)"); axes[2].set_xlabel("m")
for ax in axes:
    ax.legend()
fig.tight_layout(); fig.savefig(RUN_DIR / "height_vs_size.jpg", dpi=130); plt.show()

## 3 - Ablations: one change vs v1 (paired, bootstrap CI of the median IoU delta)

In [ ]:
def paired_delta(df, a, b="v1", col="iou"):
    d = (df[f"{a}_{col}"] - df[f"{b}_{col}"]).dropna().to_numpy()
    if len(d) < 5:
        return np.nan, (np.nan, np.nan), 0
    return round(float(np.mean(d)), 4), boot_ci(d, fn=np.mean), len(d)

abl = []
for v in VARIANTS:
    for hb in ("all", "low", "high"):
        g = ev if hb == "all" else ev[ev["height_bin"] == hb]
        mean_d, ci, n = paired_delta(g, v)
        abl.append({"variant": v, "subset": hb, "n": n,
                    "iou_med": round(g[f"{v}_iou"].median(), 3),
                    f"iou>={IOU_OK}": round((g[f"{v}_iou"] >= IOU_OK).mean(), 3),
                    "centroid_med_m": round(g[f"{v}_centroid_err_m"].median(), 3),
                    "mean_iou_delta_vs_v1": mean_d, "delta_ci": ci})
abl = pd.DataFrame(abl)
abl.to_csv(RUN_DIR / "ablations.csv", index=False)
display(abl.pivot(index="variant", columns="subset", values="mean_iou_delta_vs_v1"))
display(abl)

## 5 - v2: learned box size (5-fold cross-fitted)

Box = lateral (+1) + dimensions (along / across ego heading) = median of the
human boxes of the same object class in the OTHER folds (class-agnostic median
if the class has < 5 training sequences). `glob` uses one learned size for
every class - it separates "any sensible size" from "class-specific size".
`*_depth` puts the box's near edge on the map object instead of centring it:
on 4 hand-checked traces that HURT (the map object sits near the human box
centre - the human boxes were likely drawn over the USS map view), so centred
is the default. Folds are by sequence-id hash (stable).

In [ ]:
import hashlib

from vision_uss_research.autolabel.geometry import box_metrics
from vision_uss_research.autolabel.uss_box import box_behind_face, polygon_from_str

K_FOLDS = 5
MIN_CLASS_TRAIN = 5
g5 = ev.dropna(subset=["face_cx", "gt_len_along_m"]).copy()
g5["fold"] = g5["sequence_id"].map(lambda s: int(hashlib.md5(s.encode()).hexdigest(), 16) % K_FOLDS)
dims = ["gt_len_along_m", "gt_wid_across_m"]
prior_rows = []
for k in range(K_FOLDS):
    train, test = g5[g5["fold"] != k], g5[g5["fold"] == k]
    cls_med = train.groupby("obj")[dims].median()
    cls_n = train.groupby("obj").size()
    glob = train[dims].median()
    for _, r in test.iterrows():
        use_cls = r["obj"] in cls_med.index and cls_n[r["obj"]] >= MIN_CLASS_TRAIN
        cls_size = cls_med.loc[r["obj"]] if use_cls else glob
        gt = polygon_from_str(r["gt_polygon"])
        out = {"sequence_id": r["sequence_id"], "cls_prior_used": int(use_cls)}
        for name, size, shift in (("glob", glob, False), ("cls", cls_size, False),
                                  ("cls_depth", cls_size, True)):
            L, W = float(size.iloc[0]), float(size.iloc[1])
            corners, _ = box_behind_face((r["face_cx"], r["face_cy"]), r["ego_yaw_rad"],
                                         int(r["obj_side"]), L, W, depth_shift=shift)
            out.update({f"{name}_{m}": v for m, v in box_metrics(corners, gt).items()})
        prior_rows.append(out)
prior = pd.DataFrame(prior_rows)
ev = ev.drop(columns=[c for c in prior.columns if c != "sequence_id" and c in ev.columns])
ev = ev.merge(prior, on="sequence_id", how="left")
print(f"class prior used for {prior['cls_prior_used'].mean():.1%} of {len(prior)} sequences")

LADDER = ["v1", "lat_pos", "depth", "glob", "cls", "cls_depth"]
ladder = []
for v in LADDER:
    for hb in ("all", "low", "high"):
        g = ev if hb == "all" else ev[ev["height_bin"] == hb]
        mean_d, ci, n = paired_delta(g, v)
        ladder.append({"variant": v, "subset": hb, "n": n,
                       "iou_med": round(g[f"{v}_iou"].median(), 3),
                       "iou_med_ci": boot_ci(g[f"{v}_iou"]),
                       f"iou>={IOU_OK}": round((g[f"{v}_iou"] >= IOU_OK).mean(), 3),
                       "centroid_med_m": round(g[f"{v}_centroid_err_m"].median(), 3),
                       "corner_matched_med_m": round(g[f"{v}_corner_err_matched_m"].median(), 3),
                       "mean_iou_delta_vs_v1": mean_d, "delta_ci": ci})
ladder = pd.DataFrame(ladder)
ladder.to_csv(RUN_DIR / "v2_ladder.csv", index=False)
display(ladder.pivot(index="variant", columns="subset", values="iou_med").loc[LADDER])
display(ladder)

by_obj_v2 = quality_table(ev, prefix="cls")
by_obj_v2 = by_obj_v2.merge(by_obj[["obj", "iou_med"]].rename(columns={"iou_med": "v1_iou_med"}),
                            on="obj", how="left")
by_obj_v2.to_csv(RUN_DIR / "quality_by_object_v2.csv", index=False)
display(by_obj_v2)

## 6 - Placement: central / 45 deg / displaced

If ignoring the lateral offset is what hurts v1, `displaced` should show the
largest v1 centroid error and the largest lat_pos gain. 45 deg approaches also
rotate the object relative to the ego-aligned box.

In [ ]:
ev["approach"] = ev["approach"].fillna("").replace("", "untagged")
place = (ev.groupby(["approach", "height_bin"])
           .agg(n=("v1_iou", "size"),
                v1_iou=("v1_iou", "median"), lat_iou=("lat_pos_iou", "median"),
                v2_iou=("cls_iou", "median"),
                v1_centroid=("v1_centroid_err_m", "median"),
                lat_centroid=("lat_pos_centroid_err_m", "median"),
                v2_centroid=("cls_centroid_err_m", "median"))
           .round(3).reset_index())
place.to_csv(RUN_DIR / "by_approach.csv", index=False)
display(place)

## 4 - Failure cases for the paper figure

Worst v1 IoU per height bin, plus the object-selection failures (large centroid
error = wrong slot, not a sizing issue).

In [ ]:
cols = ["sequence_id", "obj", "height_bin", "gt_area_m2", "v1_iou", "v1_oracle_size_iou",
        "v1_centroid_err_m", "slot_v1", "slot_v2"]
for hb in ("low", "high"):
    print(f"--- worst {hb}")
    display(ev[ev["height_bin"] == hb].nsmallest(8, "v1_iou")[cols])
wrong = ev[ev["v1_centroid_err_m"] > 1.0]
print(f"wrong-object picks (centroid > 1 m): {len(wrong)}/{len(ev)} ({len(wrong)/max(len(ev),1):.1%})")
print(wrong.groupby("height_bin").size().to_string())
wrong[cols].to_csv(RUN_DIR / "wrong_object_picks.csv", index=False)

In [ ]:
summary = {
    "rows": int(len(res)), "scored": int(len(scored)), "single_target": int(len(ev)),
    "errors": res["error"].replace("", "OK").str.split(":").str[0].value_counts().to_dict(),
    "v1_iou_median": {hb: round(float(g["v1_iou"].median()), 4)
                      for hb, g in ev.groupby("height_bin")},
    "v1_oracle_size_iou_median": {hb: round(float(g["v1_oracle_size_iou"].median()), 4)
                                  for hb, g in ev.groupby("height_bin")},
    "wrong_object_rate": round(float((ev["v1_centroid_err_m"] > 1.0).mean()), 4) if len(ev) else None,
    "iou_median_ladder": {v: {hb: round(float(g[f"{v}_iou"].median()), 4)
                              for hb, g in ev.groupby("height_bin")} for v in LADDER},
}
finish_run(RUN_DIR, summary=summary)
print(_json.dumps(summary, indent=1))

## Next

- If `MAX_SEQUENCES` smoke looks sane, set it to 0 and rerun (resumes in place).
- Lateral sign settled (+1) in `al_eval_v1`; `v2` = `cls` (section 5).
- Confirm with the labelling team how Label V2 polygons were drawn (over the USS
  map view vs physical measurement) - it decides what "ground truth" means here.
- Downstream proxy: join the selected `slot_v1` object to CusReplay
  `ClassProbHigh` (alignment/uss_class.py) - does label quality predict USS
  height-classification errors?
- When second-programme access lands, rerun the same notebook on its 586 traces
  (point `RAW_ROOT_OVERRIDE` at them) for a second vehicle programme.